# 0. Imports & Reproducibility

In [4]:
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import community as community_louvain
import faiss
import networkx as nx
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from gensim.models import Word2Vec
from node2vec import Node2Vec
from torch.utils.data import DataLoader, Dataset

In [5]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 1. Download and Store Dataset

In [6]:
def ingest_reddit_data(
    subreddit_key: str, n_rows: int = 1_000_000, force_rerun: bool = False
) -> Path:
    """
    Orchestrates the ETL process for a specific subreddit's comment data.

    Args:
        subreddit_key: Dictionary key from 'splits' (e.g., 'changemyview').
        n_rows: Maximum records to process for the local sample.
        force_rerun: If True, bypasses existence check and overwrites existing parquet file.

    Returns:
        Path to the processed Parquet file.
    """
    out_path = Path(f"data/processed/{subreddit_key}_sample.parquet")

    # Idempotency check: Skip heavy network I/O if the target file is already present
    if out_path.exists() and not force_rerun:
        print(f"Skipping ingestion: Local cache found at {out_path}")
        return out_path

    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Define schema subset based on downstream analytical requirements
    feature_cols = [
        "author",
        "body",
        "created_utc",
        "id",
        "link_id",
        "name",
        "parent_id",
        "score",
        "controversiality",
        "total_awards_received",
    ]
    splits = {
        "changemyview": "data/changemyview-*-of-*.parquet",
    }

    print(f"Streaming data from HuggingFace for: r/{subreddit_key}...")

    # Execute lazy-evaluated ETL pipeline
    try:
        (
            pl.scan_parquet(
                f"hf://datasets/HuggingFaceGECLM/REDDIT_comments/{splits[subreddit_key]}"
            )
            .select(feature_cols)
            # Filter out deleted/removed content to maintain high data quality for NLP tasks
            .filter(~pl.col("body").is_in(["[deleted]", "[removed]"]))
            .limit(n_rows)
            # Stream directly to disk using ZSTD to balance compression ratio and write speed
            .sink_parquet(out_path, compression="zstd")
        )
        print(f"Successfully wrote {n_rows} rows to {out_path}")
    except KeyError:
        raise ValueError(f"Subreddit '{subreddit_key}' not found in defined splits.")
    except Exception as e:
        print(f"Pipeline failed: {e}")
        raise

    return out_path


# --- Execution Control ---
# Toggle 'force_rerun' if the upstream data schema changes or a larger sample is needed
OUT = ingest_reddit_data("changemyview", n_rows=1_000_000, force_rerun=False)

Skipping ingestion: Local cache found at data/processed/changemyview_sample.parquet


# 2. Load Data from Parquet File

In [7]:
df = (
    # Scan the metadata and define the lazy query plan
    pl.scan_parquet("data/processed/changemyview_sample.parquet")
    # Constrain sample size for rapid local prototyping
    .head(30000)
    # Trigger execution and load into memory
    .collect()
    # Bridge to Pandas for ecosystem compatibility
    .to_pandas()
)

# 3. Create Train and Test Dataset

### 3.1 Data Preprocessing, Temporal Splitting & Metadata Mapping

In [8]:
# --- 1. Data Cleaning & Type Casting ---

# Ensure text integrity by removing null observations in the primary feature
df = df.dropna(subset=["body"])

# Filter out anonymous/deleted accounts to maintain attribution quality
df = df[df["author"] != "[deleted]"]

# Normalize timestamps: Convert raw strings to numeric Unix seconds, then to datetime objects
# 'coerce' handles malformed strings by returning NaT, preventing pipeline crashes
df["created_utc"] = pd.to_numeric(df["created_utc"], errors="coerce")
df["date"] = pd.to_datetime(df["created_utc"], unit="s", errors="coerce")


# --- 2. Temporal Train/Test Split ---

# Use a temporal 80/20 split rather than a random shuffle to prevent 'look-ahead' bias.
# This simulates a real-world scenario where we predict future comments based on past data.
cutoff = df["created_utc"].quantile(0.8)

df_train = df[df["created_utc"] <= cutoff].copy()
df_test = df[df["created_utc"] > cutoff].copy()


# --- 3. Metadata Mapping (Lookup Tables) ---

# Create lightweight author lookups for efficient O(1) retrieval.
# Mappings are scoped strictly within splits to enforce isolation and prevent leakage.
id2author_train = df_train.set_index("id")["author"].to_dict()
id2author_test = df_test.set_index("id")["author"].to_dict()

### 3.2 Interaction Network Construction

In [9]:
def build_reply_pairs(df_split, id2author):
    """
    Constructs a positive interaction dataset by mapping comments to their parent authors.
    Filters for comment-to-comment replies and removes self-interactions.
    """
    # Reddit 'parent_id' prefixes: t1 = Comment, t3 = Link/Post.
    # We restrict analysis to comment-to-comment interactions to capture conversational dynamics.
    parent_comment_ids = df_split["parent_id"].astype(str)
    is_comment_reply = parent_comment_ids.str.startswith("t1_")
    df_r = df_split[is_comment_reply].copy()

    # Extract the raw 36-base ID by stripping the 't1_' type prefix for join compatibility
    df_r["parent_key"] = df_r["parent_id"].str.replace("^t1_", "", regex=True)

    # Resolve parent author identities via the provided lookup table (O(1) mapping)
    df_r["parent_author"] = df_r["parent_key"].map(id2author)

    # --- Data Integrity & Quality Filtering ---
    # 1. Drop replies where the parent comment falls outside the current split (boundary integrity)
    df_r = df_r.dropna(subset=["parent_author"])
    # 2. Exclude self-replies to ensure we only model interpersonal interactions
    df_r = df_r[df_r["author"] != df_r["parent_author"]]

    # Feature selection and renaming to standard (u, v) graph notation
    pairs_pos = df_r[
        ["author", "parent_author", "created_utc", "link_id", "id", "parent_key"]
    ].copy()
    pairs_pos = pairs_pos.rename(
        columns={
            "author": "u",
            "parent_author": "v",
            "id": "u_comment_id",
            "parent_key": "v_comment_id",
        }
    )

    # Label as positive instances for downstream binary classification
    pairs_pos["y"] = 1
    return pairs_pos


# Generate interaction sets; scoped within splits to prevent data leakage
pos_train = build_reply_pairs(df_train, id2author_train)
pos_test = build_reply_pairs(df_test, id2author_test)

print(f"Positive samples - Train: {len(pos_train):,} | Test: {len(pos_test):,}")

Positive samples - Train: 12,808 | Test: 3,114


In [10]:
# --- Graph Diagnostics: Sparsity & Degree Distribution ---

# Calculate the ratio of users who engaged in at least one reply
pos_users = set(pos_train["u"]) | set(pos_train["v"])
all_users = set(df_train["author"].dropna().unique())
print(
    f"Engagement Coverage: {len(pos_users)} / {len(all_users)} users with interactions"
)

# Analyze the 'Out-Degree' (number of replies sent per user)
print("\nReplies per user statistics:")
print(pos_train.groupby("u").size().describe())

Engagement Coverage: 2502 / 3265 users with interactions

Replies per user statistics:
count    2249.000000
mean        5.694976
std        16.164946
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max       467.000000
dtype: float64


### 3.3 Negative Sampling Strategy

In [11]:
def build_hard_negatives(df_split, pos_pairs, k_per_pos=2, seed=42):
    """
    Generates 'hard' negative samples for link prediction by identifying potential
    interactions that did NOT occur within the same discussion thread context.
    """
    # Initialize a BitGenerator for reproducible stochastic sampling
    rng = np.random.default_rng(seed)

    # 1) Contextual Mapping: Identify all active participants per discussion thread (link_id).
    # This defines our 'closed-world' candidate pool for each observation.
    thread_users = (
        df_split.groupby("link_id")["author"].apply(lambda s: set(s.dropna())).to_dict()
    )

    # 2) Network Topology: Extract existing interaction edges in (u, v) space.
    # We treat edges as symmetric to prevent sampling reciprocal replies as negatives,
    # which would introduce label noise.
    reply_edges = set(zip(pos_pairs["u"], pos_pairs["v"]))
    reply_edges_sym = reply_edges | {(v, u) for (u, v) in reply_edges}

    neg_rows = []
    # Project to minimal feature set to reduce overhead during iteration
    pos_pairs_small = pos_pairs[["u", "v", "link_id"]].copy()

    for u, v, link_id in pos_pairs_small.itertuples(index=False):
        users = list(thread_users.get(link_id, []))
        if len(users) <= 1:
            continue

        # Candidate Filtering:
        # Target users in the same thread (high-signal 'hard' negatives) excluding the source 'u'
        cand = [x for x in users if x != u]
        if not cand:
            continue

        # Collision Avoidance: Remove candidates where a ground-truth interaction (u, x) exists
        cand = [x for x in cand if (u, x) not in reply_edges_sym]
        if not cand:
            continue

        # Stochastic Sampling: Select 'k' negatives per positive to maintain class ratio
        take = min(k_per_pos, len(cand))
        sampled = rng.choice(cand, size=take, replace=False)

        for x in sampled:
            neg_rows.append((u, x, link_id, 0))

    return pd.DataFrame(neg_rows, columns=["u", "v", "link_id", "y"])


# --- Triplet Dataset Assembly ---


def build_triplets_from_hard_negatives(pos_pairs, neg_pairs, seed=42):
    """
    Constructs triplets (u, v_pos, v_neg) for metric learning.
    For each positive interaction (u, v_pos), sample one hard negative v_neg
    from the same thread context.
    """
    rng = np.random.default_rng(seed)

    # Map u → list of negative candidates
    neg_map = neg_pairs.groupby("u")["v"].apply(list).to_dict()

    triplets = []

    for u, v_pos, link_id in pos_pairs[["u", "v", "link_id"]].itertuples(index=False):
        neg_candidates = neg_map.get(u, [])
        if not neg_candidates:
            continue

        # Sample one hard negative for this positive
        v_neg = rng.choice(neg_candidates)

        triplets.append((u, v_pos, v_neg, link_id))

    return pd.DataFrame(triplets, columns=["u", "v_pos", "v_neg", "link_id"])


# --- Generate hard negatives (unchanged) ---
neg_train = build_hard_negatives(df_train, pos_train, k_per_pos=2)
neg_test = build_hard_negatives(df_test, pos_test, k_per_pos=2)

# --- Build triplets ---
triplets_train = build_triplets_from_hard_negatives(pos_train, neg_train)
triplets_test = build_triplets_from_hard_negatives(pos_test, neg_test)

print("Triplets Train:", len(triplets_train))
print("Triplets Test:", len(triplets_test))
print(triplets_train.head())


Triplets Train: 12747
Triplets Test: 2944
                 u                 v_pos            v_neg    link_id
0        Jaberkaty  Thompson_S_Sweetback  ancillarynipple  t3_16ralh
1  ancillarynipple  Thompson_S_Sweetback        Jaberkaty  t3_16ralh
2  ancillarynipple  Thompson_S_Sweetback        Jaberkaty  t3_16ralh
3           llatia             gchase723            xmreg  t3_16s6jg
4     cardswsbound                 nix0n           llatia  t3_16rzx1


In [12]:
def test_triplet_integrity(triplets, pos_df, neg_df):
    """
    Comprehensive suite to verify triplet logic and data leakage.
    """
    # 1. Structural Check
    assert not triplets.isnull().values.any(), "Triplets contain NaN values"

    # 2. Contextual Integrity: v_neg must actually exist in the negative pool for that user
    # This ensures rng.choice didn't pull a random user from the wrong context
    u_to_negs = neg_df.groupby("u")["v"].apply(set).to_dict()

    for row in triplets.itertuples():
        # Check: v_pos and v_neg must be different
        assert row.v_pos != row.v_neg, f"Anchor {row.u} has identical Pos/Neg target"

        # Check: anchor cannot be its own target
        assert row.u != row.v_pos, f"Self-loop found in positive: {row.u}"
        assert row.u != row.v_neg, f"Self-loop found in negative: {row.u}"

        # Check: v_neg must be a valid 'hard' negative from the pool
        valid_negs = u_to_negs.get(row.u, set())
        assert row.v_neg in valid_negs, (
            f"User {row.v_neg} is not a valid hard negative for {row.u}"
        )

    # 3. Label Leakage: Ensure v_neg is NEVER a real positive for that user
    # (Symmetric check to be extra safe)
    pos_edges = set(zip(pos_df["u"], pos_df["v"]))
    pos_edges_sym = pos_edges | {(v, u) for (u, v) in pos_edges}

    triplet_neg_edges = set(zip(triplets["u"], triplets["v_neg"]))
    overlap = triplet_neg_edges.intersection(pos_edges_sym)

    assert len(overlap) == 0, (
        f"Leakage detected! {len(overlap)} 'negatives' are actually real interactions."
    )

    print("✅ All Triplet Integrity Tests Passed!")


# Run the test
test_triplet_integrity(triplets_train, pos_train, neg_train)

✅ All Triplet Integrity Tests Passed!


### 3.4 User Textual Profile Construction

In [13]:
# --- 1. Corpus Preparation & Leakage Prevention ---

# Isolate training and testing text to ensure that future comments do not
# influence the historical representations of users in the training set.
df_train_text = df_train.dropna(subset=["body", "id"]).copy()
df_test_text = df_test.dropna(subset=["body", "id"]).copy()

# (Optional) Heuristic: Filter for active users to ensure embeddings have
# sufficient signal (min 5 observations).
# df_train_text = df_train_text.groupby("id").filter(lambda g: len(g) >= 5)

# --- 2. Temporal Aggregation (Feature Engineering) ---

# Construct a profile for each author.
# We join the most recent comments to capture the user's current interests/voice.
user_text_train = (
    df_train_text.sort_values(
        "created_utc"
    )  # Enforce chronology to correctly identify the 'tail'
    .groupby("author")["body"]
    # Hyperparameter: Concatenating the last 10 comments balances context vs. sequence length
    .apply(lambda s: " ".join(s.tail(10)))
)

user_text_test = (
    df_test_text.sort_values("created_utc")
    .groupby("author")["body"]
    .apply(lambda s: " ".join(s.tail(10)))
)

# Convert to hash maps (dict) for O(1) lookup performance during the mapping phase
user_text_dict_train = user_text_train.to_dict()
user_text_dict_test = user_text_test.to_dict()

# --- 3. Coverage Analysis (Data Integrity Check) ---

# Quantify the 'Cold-Start' issue: users in the interaction pairs who lack
# textual history. Significant missingness here indicates a sampling mismatch.
missing_train = triplets_train["u"].map(user_text_dict_train).isna().mean()
print(f"Missing text profile ratio (Train - Source User): {missing_train:.2%}")

missing_test = triplets_test["u"].map(user_text_dict_test).isna().mean()
print(f"Missing text profile ratio (Test - Source User): {missing_test:.2%}")

Missing text profile ratio (Train - Source User): 0.00%
Missing text profile ratio (Test - Source User): 0.00%


In [14]:
# --- 1. Prepare Data Containers ---

# Create copies to prevent SettingWithCopy warnings and isolate split changes
triplets_train = triplets_train.copy()
triplets_test = triplets_test.copy()


def attach_text(triplets, user_text_dict):
    """Adds historical text for source (u) and target (v) users."""

    # Map text profiles to user IDs
    triplets["text_u"] = triplets["u"].map(user_text_dict)
    triplets["text_v_pos"] = triplets["v_pos"].map(user_text_dict)
    triplets["text_v_neg"] = triplets["v_neg"].map(user_text_dict)

    # Remove observations missing text for either user to ensure a complete feature set
    return triplets.dropna(subset=["text_u", "text_v_pos", "text_v_neg"])


# --- 2. Execute Merge & Cleanup ---

triplets_train_txt = attach_text(triplets_train, user_text_dict_train)
triplets_test_txt = attach_text(triplets_test, user_text_dict_test)

# --- 3. Progress Check ---

# Log row counts to monitor data loss during the mapping/dropping process
print(f"Train Retention: {len(triplets_train):,} -> {len(triplets_train_txt):,}")
print(f"Test Retention:  {len(triplets_test):,} -> {len(triplets_test_txt):,}")

Train Retention: 12,747 -> 12,747
Test Retention:  2,944 -> 2,944


### 3.5 Save Train and Test Datasets

In [15]:
triplets_train_txt.to_parquet("data/processed/train_triplets_txt.parquet", index=False)
triplets_test_txt.to_parquet("data/processed/test_triplets_txt.parquet", index=False)

# 4.  Implement Baselines

### 4.1 Random Baseline

In [16]:
# 1. Candidates = all users from training
all_users_train = set(pos_train["u"]).union(set(pos_train["v"]))
all_users_train = list(all_users_train)

def recommend_random(user, k=10, exclude_seen=True):
    """
    Recommend k random users.
    
    Args:
        user: source user
        k: number of recommendations
        exclude_seen: avoid recommending already interacted users (train)
    """
    if not exclude_seen:
        candidates = [u for u in all_users_train if u != user]
        return random.sample(candidates, k)

    # Users already interacted with in training
    seen = set(pos_train[pos_train["u"] == user]["v"].values)

    candidates = [
        u for u in all_users_train
        if u != user and u not in seen
    ]

    if len(candidates) < k:
        return candidates

    return random.sample(candidates, k)

### 4.2 Common Neighbor Baseline

In [17]:
# 1. Build an Adjacency List from the TRAIN set
# We treat the network as undirected to find "friends of friends" (mutual interactors)
adj = defaultdict(set)
for u, v in zip(pos_train["u"], pos_train["v"]):
    adj[u].add(v)
    adj[v].add(u)

def recommend_common_neighbors(user, k=10, exclude_seen=True):
    """
    Recommend users based on the number of shared interaction partners (Common Neighbors).
    
    Args:
        user: The source user for whom to generate recommendations.
        k: Number of recommendations to return.
        exclude_seen: If True, prevents recommending users already interacted with in training.
    """
    if user not in adj:
        # Cold-start: If the user has no history, no neighbors can be found
        return []

    user_neighbors = adj[user]
    candidate_scores = defaultdict(int)
    
    # Traverse to neighbors (friends) and then to their neighbors (friends of friends)
    for neighbor in user_neighbors:
        for fof in adj[neighbor]: # Friend of a Friend
            if fof != user:
                # Increment score for every shared path (common neighbor)
                candidate_scores[fof] += 1
    
    # Filter: Remove users the target user has already interacted with in the training set
    if exclude_seen:
        for seen_user in user_neighbors:
            if seen_user in candidate_scores:
                del candidate_scores[seen_user]
                
    # Sort candidates by the number of common neighbors in descending order
    sorted_recs = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)
    return [rec[0] for rec in sorted_recs[:k]]

### 4.3 Popularity based Baseline

In [18]:
# 1. Compute popularity scores from TRAIN only
popularity_scores = (
    pos_train["v"]
    .value_counts()
    .to_dict()
)

# 2. Global ranking of users by popularity
global_pop_ranking = sorted(
    popularity_scores.keys(),
    key=lambda x: popularity_scores[x],
    reverse=True
)


def recommend_popularity(user, k=10, exclude_seen=True):
    """
    Recommend top-k most popular users.
    
    Args:
        user: source user
        k: number of recommendations
        exclude_seen: avoid recommending already interacted users (train)
    """
    if not exclude_seen:
        return global_pop_ranking[:k]

    # Users already interacted with in training
    seen = set(pos_train[pos_train["u"] == user]["v"].values)

    recs = []
    for candidate in global_pop_ranking:
        if candidate != user and candidate not in seen:
            recs.append(candidate)
        if len(recs) == k:
            break

    return recs

# 5. Train CNN

### 5.1 Create Vocabulary

In [19]:
# Define a simple regex to extract word-level tokens (alphabetic only)
TOKEN_RE = re.compile(r"[A-Za-z']+")


def tokenize(text: str):
    """Lowercases and extracts valid word tokens from raw text."""
    return TOKEN_RE.findall(text.lower())


# Constraints for memory efficiency and noise reduction
MAX_VOCAB = 50_000
MIN_FREQ = 2

# Build frequency distribution from training corpus only to prevent leakage
counter = Counter()
for t in triplets_train_txt["text_u"].tolist():
    counter.update(tokenize(t))
for t in triplets_train_txt["text_v_pos"].tolist():
    counter.update(tokenize(t))
for t in triplets_train_txt["text_v_neg"].tolist():
    counter.update(tokenize(t))

# Reserved tokens for sequence padding and out-of-vocabulary terms
PAD = "<pad>"
UNK = "<unk>"

# Initialize vocabulary with reserved indices
vocab = {PAD: 0, UNK: 1}

# Populate vocabulary with the most frequent terms meeting the frequency threshold
for w, c in counter.most_common(MAX_VOCAB):
    if c < MIN_FREQ:
        break
    vocab[w] = len(vocab)

pad_id = vocab[PAD]
unk_id = vocab[UNK]

print(f"Final Vocab Size: {len(vocab):,}")

Final Vocab Size: 30,550


### 5.2 Dataset Definition & User Text Encoding

In [20]:
# Fixed sequence length to ensure uniform input dimensions for the model
MAX_LEN = 256


def encode(text: str):
    """
    Converts raw text into a list of integer IDs.
    Unknown words are mapped to 'unk_id' and sequences are truncated to MAX_LEN.
    """
    ids = [vocab.get(w, unk_id) for w in tokenize(text)]
    return ids[:MAX_LEN]


class TripletDataset(Dataset):
    """
    Custom PyTorch Dataset to serve (u, v_pos, v_neg) triplets, their respective
    encoded histories, and the binary interaction label.
    """

    def __init__(self, df_triplets):
        self.u = df_triplets["u"].tolist()
        self.v_pos = df_triplets["v_pos"].tolist()
        self.v_neg = df_triplets["v_neg"].tolist()
        self.u_texts = df_triplets["text_u"].tolist()
        self.v_pos_texts = df_triplets["text_v_pos"].tolist()
        self.v_neg_texts = df_triplets["text_v_neg"].tolist()

    def __len__(self):
        return len(self.u)

    def __getitem__(self, idx):
        # Returns raw IDs and encoded text sequences for the given index
        return (
            self.u[idx],
            self.v_pos[idx],
            self.v_neg[idx],
            encode(self.u_texts[idx]),
            encode(self.v_pos_texts[idx]),
            encode(self.v_neg_texts[idx]),
        )

In [21]:
def collate_fn(batch):
    """
    Dynamic padding: Aligns sequences within a batch to the length of
    the longest sequence found in that specific batch.
    """
    # Unpack columns from the batch of tuples
    u, v_pos, v_neg, u_seqs, v_pos_seqs, v_neg_seqs = zip(*batch)

    # Track original lengths for masking or sequence packing
    u_lens = torch.tensor([len(s) for s in u_seqs], dtype=torch.long)
    v_pos_lens = torch.tensor([len(s) for s in v_pos_seqs], dtype=torch.long)
    v_neg_lens = torch.tensor([len(s) for s in v_neg_seqs], dtype=torch.long)

    # Determine batch-wide maximum dimensions
    max_u = max(u_lens).item()
    max_v_pos = max(v_pos_lens).item()
    max_v_neg = max(v_neg_lens).item()

    # Initialize tensors filled with the PAD token
    u_tensor = torch.full((len(batch), max_u), pad_id, dtype=torch.long)
    v_pos_tensor = torch.full((len(batch), max_v_pos), pad_id, dtype=torch.long)
    v_neg_tensor = torch.full((len(batch), max_v_neg), pad_id, dtype=torch.long)

    # Copy sequence data into the padded containers
    for i, s in enumerate(u_seqs):
        u_tensor[i, : len(s)] = torch.tensor(s, dtype=torch.long)

    for i, s in enumerate(v_pos_seqs):
        v_pos_tensor[i, : len(s)] = torch.tensor(s, dtype=torch.long)

    for i, s in enumerate(v_neg_seqs):
        v_neg_tensor[i, : len(s)] = torch.tensor(s, dtype=torch.long)

    return (
        list(u),
        list(v_pos),
        list(v_neg),
        u_tensor,
        v_pos_tensor,
        v_neg_tensor,
        u_lens,
        v_pos_lens,
        v_neg_lens,
    )

### 5.3 Data Loader Initilization

In [22]:
# Number of samples processed before the model updates its internal parameters
BATCH_SIZE = 128

# Instantiate dataset objects for training and evaluation
train_ds = TripletDataset(triplets_train_txt)
test_ds = TripletDataset(triplets_test_txt)

# Training Loader: Shuffle enabled to prevent the model from learning the order of samples
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, generator=torch.Generator().manual_seed(SEED)
)

# Testing Loader: Shuffle disabled to ensure consistent, reproducible evaluation
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, generator=torch.Generator().manual_seed(SEED)
)

### 5.4 Create Siamese CNN

##### 5.4.1 Create Text Encoder to build user embeddings

In [23]:
class TextCNNEncoder(nn.Module):
    """
    Multi-kernel CNN for extracting hierarchical n-gram features from text.
    Outputs a normalized embedding representing a user's linguistic style.
    """

    def __init__(
        self,
        vocab_size,
        emb_dim=128,
        num_filters=128,
        kernel_sizes=(3, 4, 5),
        out_dim=128,
        pad_idx=0,
        dropout=0.2,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)

        # Parallel convolutional layers to capture varied phrase lengths
        self.convs = nn.ModuleList(
            [
                nn.Conv1d(in_channels=emb_dim, out_channels=num_filters, kernel_size=k)
                for k in kernel_sizes
            ]
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), out_dim)

    def forward(self, x):
        # x: [Batch, Sequence_Length]
        emb = self.embedding(x)
        emb = emb.transpose(1, 2)  # Align dimensions for 1D convolution [B, E, T]

        conv_outs = []
        for conv in self.convs:
            # Apply convolution and non-linearity
            h = F.relu(conv(emb))
            # Global Max Pooling: Extract the most salient feature per filter
            h = F.max_pool1d(h, kernel_size=h.size(2)).squeeze(2)
            conv_outs.append(h)

        # Fusion of multi-scale features
        h = torch.cat(conv_outs, dim=1)
        h = self.dropout(h)
        h = self.fc(h)

        # L2 Normalization to facilitate cosine similarity downstream
        h = F.normalize(h, p=2, dim=1)
        return h

##### 5.4.2 Create Siamese Architecture

In [24]:
class SiameseCNN(nn.Module):
    """
    Siamese architecture for metric learning.
    Uses a shared encoder to map the anchor, positive, and negative
    samples into a common vector space.
    """

    def __init__(self, encoder: nn.Module):
        super().__init__()
        # The core Siamese principle: one encoder, shared weights.
        self.encoder = encoder

    def forward(self, u_tensor, v_pos_tensor, v_neg_tensor):
        # Pass all three through the identical encoder
        # This maps them to the same latent space for distance comparison
        emb_u = self.encoder(u_tensor)
        emb_pos = self.encoder(v_pos_tensor)
        emb_neg = self.encoder(v_neg_tensor)

        return emb_u, emb_pos, emb_neg

##### 5.4.3 Model Instatntiation & Device Allocation

In [25]:
# Automatically detect if a GPU is available for accelerated training
device = torch.device("mps" if torch.mps.is_available() else "cpu")

# Initialize the Feature Extractor (Encoder)
# We use a Multi-Kernel CNN to capture n-gram patterns of lengths 3, 4, and 5
encoder = TextCNNEncoder(
    vocab_size=len(vocab),  # Determined by the tokenizer in Section 6
    emb_dim=128,  # Dimensionality of the dense word vectors
    num_filters=128,  # Number of features to extract per kernel size
    kernel_sizes=(3, 4, 5),  # Window sizes: Tri-grams, 4-grams, 5-grams
    out_dim=128,  # Final embedding size (compact semantic vector)
    pad_idx=pad_id,  # Index to ignore during embedding lookup (zero gradient)
    dropout=0.4,  # Regularization to prevent overfitting on specific phrases
)

# Wrap the encoder in the Siamese architecture for metric learning
# scale=10.0 expands the cosine range [-1, 1] to [-10, 10] for sharper probability gradients
model = SiameseCNN(encoder).to(device)

print(f"Model initialized on: {device}")

Model initialized on: mps


### 5.5 Train the CNN

In [26]:
# 1. New Criterion for Metric Learning
# p=2 indicates Euclidean distance. With your L2-normalized CNN,
# this focuses the learning on the angular separation between users.
criterion = nn.TripletMarginLoss(margin=1.0, p=2)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)


@torch.no_grad()
def eval_triplet_acc(model, loader):
    model.eval()
    correct = 0
    total = 0

    # Unpacking the three context-aware tensors
    for _, _, _, u_tensor, v_pos_tensor, v_neg_tensor, _, _, _ in loader:
        u_tensor = u_tensor.to(device)
        v_pos_tensor = v_pos_tensor.to(device)
        v_neg_tensor = v_neg_tensor.to(device)

        # Get embeddings from SiameseCNN
        emb_u, emb_v_pos, emb_v_neg = model(u_tensor, v_pos_tensor, v_neg_tensor)

        # Calculate Euclidean distances: d(anchor, positive) vs d(anchor, negative)
        dist_pos = torch.norm(emb_u - emb_v_pos, p=2, dim=1)
        dist_neg = torch.norm(emb_u - emb_v_neg, p=2, dim=1)

        # A successful "ranking" occurs when the positive is closer than the hard negative
        correct += (dist_pos < dist_neg).sum().item()
        total += u_tensor.size(0)

    return correct / total if total > 0 else 0.0


def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0

    for _, _, _, u_tensor, v_pos_tensor, v_neg_tensor, _, _, _ in loader:
        u_tensor = u_tensor.to(device)
        v_pos_tensor = v_pos_tensor.to(device)
        v_neg_tensor = v_neg_tensor.to(device)

        optimizer.zero_grad(set_to_none=True)

        # Forward pass through shared Siamese weights
        emb_u, emb_v_pos, emb_v_neg = model(u_tensor, v_pos_tensor, v_neg_tensor)

        # The loss pulls u closer to v_pos and pushes it away from v_neg
        loss = criterion(emb_u, emb_v_pos, emb_v_neg)

        loss.backward()
        # Gradient clipping prevents the "exploding gradient" problem in deep CNNs
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * u_tensor.size(0)

    return total_loss / len(loader.dataset)


# --- Run Loop ---
EPOCHS = 5
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model, train_loader)

    # Accuracy is the % of triplets correctly ranked
    test_acc = eval_triplet_acc(model, test_loader)
    train_acc = eval_triplet_acc(model, train_loader)

    print(
        f"Epoch {epoch:02d} | Loss: {loss:.4f} | Train Acc: {train_acc:.2%} | Test Acc: {test_acc:.2%}"
    )

Epoch 01 | Loss: 0.9577 | Train Acc: 72.20% | Test Acc: 60.60%
Epoch 02 | Loss: 0.8313 | Train Acc: 72.53% | Test Acc: 60.97%
Epoch 03 | Loss: 0.7601 | Train Acc: 73.26% | Test Acc: 58.83%
Epoch 04 | Loss: 0.7302 | Train Acc: 74.09% | Test Acc: 59.88%
Epoch 05 | Loss: 0.7033 | Train Acc: 74.42% | Test Acc: 60.33%


### 5.6 Export the User "Galaxy" (Embeddings)

In [27]:
@torch.no_grad()
def generate_user_embeddings_batched(model, unique_user_df, batch_size=256):
    model.eval()
    user_ids = unique_user_df["author"].tolist()

    # 1. Pre-numericalize all texts using your existing 'encode' function
    # We pad them manually here to create a valid tensor
    all_seqs = [encode(t) for t in unique_user_df["body"]]
    max_len = max(len(s) for s in all_seqs)

    padded_seqs = torch.full((len(all_seqs), max_len), pad_id, dtype=torch.long)
    for i, s in enumerate(all_seqs):
        padded_seqs[i, : len(s)] = torch.tensor(s)

    # 2. Extract embeddings in batches using the GPU
    all_embs = []
    for i in range(0, len(padded_seqs), batch_size):
        batch = padded_seqs[i : i + batch_size].to(device)
        # Use ONLY the encoder; we don't need the Siamese wrapper for single users
        emb = model.encoder(batch)
        all_embs.append(emb.cpu().numpy())

    full_matrix = np.vstack(all_embs)
    return {u_id: vec for u_id, vec in zip(user_ids, full_matrix)}, full_matrix


# --- How to call it ---
# Create the unique user list from your test or train data
unique_users = (
    df_train.groupby("author")["body"]
    .apply(lambda s: " ".join(s.tail(10)))
    .reset_index()
)

user_vectors, embeddings_matrix = generate_user_embeddings_batched(model, unique_users)

text_embeddings = user_vectors

# 6. Create Graph Embeddings

In [28]:
# ======================================================
# 5.X Train Node2Vec on directed training graph
# ======================================================

user_id_list = unique_users["author"].tolist()


# Build directed graph from training interactions
G = nx.DiGraph()

for u, v in pos_train[["u", "v"]].itertuples(index=False):
    G.add_edge(u, v)

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())

# Train Node2Vec
node2vec = Node2Vec(
    G,
    dimensions=64,
    walk_length=30,
    num_walks=200,
    workers=1,
    p=1.0,
    q=1.0,
    seed=42
)


n2v_model = Word2Vec(
    node2vec.walks, 
    vector_size=64, 
    window=10, 
    min_count=1, 
    batch_words=128,
    seed=42, 
    workers=1
)

# Extract graph embeddings aligned with text embeddings
graph_dim = 64
graph_embeddings = {}

for node in G.nodes():
    vec = n2v_model.wv[str(node)]
    
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec = vec / norm
        
    graph_embeddings[node] = vec

Graph nodes: 2502
Graph edges: 7921


Computing transition probabilities:   0%|          | 0/2502 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|██████████| 200/200 [00:19<00:00, 10.11it/s]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


# 7. Evaluation

### 7.1 Build Ground Truth

In [29]:
# 1. Build neighbor dictionaries
train_neighbors = pos_train.groupby("u")["v"].apply(set).to_dict()
test_neighbors = pos_test.groupby("u")["v"].apply(set).to_dict()

# 2. Users that exist in embedding index
embedded_users = set(user_vectors.keys())

ground_truth = {}

for u in test_neighbors:
    # Skip users without embeddings (cannot generate recommendations)
    if u not in embedded_users:
        continue

    train_set = train_neighbors.get(u, set())

    # Remove already seen interactions (only new links)
    new_interactions = test_neighbors[u] - train_set

    # Keep only targets that also have embeddings
    new_interactions = {v for v in new_interactions if v in embedded_users}

    # Only keep users with at least one evaluable target
    if len(new_interactions) > 0:
        ground_truth[u] = new_interactions


In [30]:
# Number of valid future targets per user
gt_sizes = {u: len(vs) for u, vs in ground_truth.items()}

# Convert to DataFrame for easier inspection
gt_df = pd.DataFrame.from_dict(gt_sizes, orient="index", columns=["n_targets"])
gt_df.index.name = "user"

gt_df.head()


,n_targets
user,
1r0n1k,2
25X,2
294116002,7
2shy2talk,1
2xwhyzed,2


### 7.2 Evaluate Precision@K

In [31]:
def evaluate_precision_at_k(model_recommend_fn, ground_truth, k=10):
    """
    model_recommend_fn: function(user_id, k) -> list of recommended users
    ground_truth: dict {u: set(valid target users)}
    """
    total_hits = 0
    total_users = 0

    for u, true_targets in ground_truth.items():
        recs = model_recommend_fn(u, k=k)

        # Safety: ensure only evaluable users are recommended
        recs = [r for r in recs if r in user_vectors]

        hits = len(set(recs) & true_targets)

        total_hits += hits
        total_users += 1

    if total_users == 0:
        return 0.0

    precision = total_hits / (k * total_users)
    return precision

In [48]:
precision_random = evaluate_precision_at_k(recommend_random, ground_truth, k=10)
precision_popularity = evaluate_precision_at_k(recommend_popularity, ground_truth, k=10)
precision_common_neighbors = evaluate_precision_at_k(recommend_common_neighbors, ground_truth, k=10)

print("Random Baseline Precision@10:", precision_random)
print("Popularity Baseline Precision@10:", precision_popularity)
print("Common Neighbors Baseline Precision@10:", precision_common_neighbors)

Random Baseline Precision@10: 0.0003816793893129771
Popularity Baseline Precision@10: 0.01984732824427481
Common Neighbors Baseline Precision@10: 0.008778625954198474


### 7.3 Evaluate Recall@K

In [33]:
def evaluate_recall_at_k(model_recommend_fn, ground_truth, k=10):
    """
    model_recommend_fn: function(user_id, k) -> list of recommended users
    ground_truth: dict {u: set(valid target users)}
    """

    total_recall = 0.0
    total_users = 0

    for u, true_targets in ground_truth.items():
        recs = model_recommend_fn(u, k=k)

        # Safety: ensure candidate universe consistency
        recs = [r for r in recs if r in user_vectors]

        hits = len(set(recs) & true_targets)

        recall_u = hits / len(true_targets)

        total_recall += recall_u
        total_users += 1

    if total_users == 0:
        return 0.0

    return total_recall / total_users

In [49]:
recall_random = evaluate_recall_at_k(
    recommend_random,
    ground_truth,
    k=10
)

recall_popularity = evaluate_recall_at_k(
    recommend_popularity,
    ground_truth,
    k=10
)

recall_common_neighbors = evaluate_recall_at_k(
    recommend_common_neighbors,
    ground_truth,
    k=10
)

print("Random Baseline Recall@10:", recall_random)
print("Popularity Baseline Recall@10:", recall_popularity)
print("Common Neighbors Baseline Recall@10:", recall_common_neighbors)

Random Baseline Recall@10: 0.0
Popularity Baseline Recall@10: 0.09852284394771335
Common Neighbors Baseline Recall@10: 0.03541851824796782


### 7.4 Evaluate nDCG@K

In [35]:
def evaluate_ndcg_at_k(model_recommend_fn, ground_truth, k=10):
    """
    model_recommend_fn: function(user_id, k) -> ranked list
    ground_truth: dict {u: set(valid targets)}
    """
    
    total_ndcg = 0.0
    total_users = 0

    for u, true_targets in ground_truth.items():
        recs = model_recommend_fn(u, k=k)

        # Compute DCG
        dcg = 0.0
        for rank, candidate in enumerate(recs, start=1):
            if candidate in true_targets:
                dcg += 1.0 / np.log2(rank + 1)

        # Compute IDCG
        ideal_hits = min(len(true_targets), k)
        idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))

        if idcg == 0:
            continue  # skip users with no valid ground truth (safety)

        ndcg_u = dcg / idcg

        total_ndcg += ndcg_u
        total_users += 1

    if total_users == 0:
        return 0.0

    return total_ndcg / total_users

In [50]:
ndcg_random = evaluate_ndcg_at_k(
    recommend_random,
    ground_truth,
    k=10
)

ndcg_popularity = evaluate_ndcg_at_k(
    recommend_popularity,
    ground_truth,
    k=10
)

ndcg_common_neighbors = evaluate_ndcg_at_k(
    recommend_common_neighbors,
    ground_truth,
    k=10
)

print("Random Baseline nDCG@10:", ndcg_random)
print("Popularity Baseline nDCG@10:", ndcg_popularity)
print("Common Neighbors Baseline nDCG@10:", ndcg_common_neighbors)

Random Baseline nDCG@10: 0.003436764743124222
Popularity Baseline nDCG@10: 0.06378789712107362
Common Neighbors Baseline nDCG@10: 0.024313039763936475


### 7.5 Evaluate Echo Chamber Metrics

##### 7.5.1 Defining individual diversity and novelty functions

In [37]:
dm_map = {}
for user in user_id_list:
    if user in graph_embeddings and user in text_embeddings:
        # L2-Normalize each part individually to balance their influence
        g_norm = graph_embeddings[user] / (np.linalg.norm(graph_embeddings[user]) + 1e-9)
        t_norm = text_embeddings[user] / (np.linalg.norm(text_embeddings[user]) + 1e-9)
        # Result is a 192-dim unified vector
        dm_map[user] = np.concatenate([g_norm, t_norm])
        dm_map[user] = dm_map[user] / np.linalg.norm(dm_map[user])

def get_dm(u, v):
    return np.linalg.norm(dm_map[u] - dm_map[v])

train_edges = defaultdict(set)

for u, v in pos_train[["u", "v"]].itertuples(index=False):
    train_edges[u].add(v)   # only outgoing

In [38]:
def calculate_dm_distance(u, v):
    return np.linalg.norm(dm_map[u] - dm_map[v])

# Individual Diversity
def individual_diversity(recs):
    # Average dm between all pairs in the Top-10
    dists = [calculate_dm_distance(i, j) for i in recs for j in recs if i != j]
    return np.mean(dists) if dists else 0

# Individual Novelty
def individual_novelty(recs, existing_interactions):
    # Average dm between Top-10 and the Training set (Fu)
    dists = [calculate_dm_distance(r, f) for r in recs for f in existing_interactions]
    return np.mean(dists) if dists else 0

##### 7.5.2 Evaluate Individual Diversity @ k

In [39]:
def evaluate_individual_diversity_at_k(model_recommend_fn, k=10):
    total_div = 0.0
    total_users = 0

    for u in ground_truth.keys():
        recs = model_recommend_fn(u, k=k)

        # Only consider users with at least 2 recommendations
        if len(recs) < 2:
            continue

        dists = []

        for i in range(len(recs)):
            for j in range(i + 1, len(recs)):
                if recs[i] in dm_map and recs[j] in dm_map:
                    d = calculate_dm_distance(recs[i], recs[j])
                    dists.append(d)

        if len(dists) == 0:
            continue

        total_div += np.mean(dists)
        total_users += 1

    return total_div / total_users if total_users > 0 else 0.0

In [51]:
indiv_div_random = evaluate_individual_diversity_at_k(
    recommend_random,
    k=10
)

indiv_div_popularity = evaluate_individual_diversity_at_k(
    recommend_popularity,
    k=10
)

indiv_div_common_neighbors = evaluate_individual_diversity_at_k(
    recommend_common_neighbors,
    k=10
)

print("Random Baseline Individual Diversity@10:", indiv_div_random)
print("Popularity Baseline Individual Diversity@10:", indiv_div_popularity)
print("Common Neighbors Baseline Individual Diversity@10:", indiv_div_common_neighbors)

Random Baseline Individual Diversity@10: 1.294357898581119
Popularity Baseline Individual Diversity@10: 1.11380312028732
Common Neighbors Baseline Individual Diversity@10: 1.16464300547974


##### 7.5.3 Evaluate Individual Novelty @ k

In [41]:
def evaluate_individual_novelty_at_k(model_recommend_fn, k=10):
    total_novelty = 0.0
    total_users = 0

    for u in ground_truth.keys():
        recs = model_recommend_fn(u, k=k)

        # Training neighbors (historical interactions)
        F_u = train_edges.get(u, set())

        # Need at least 1 recommendation and 1 historical neighbor
        if len(recs) == 0 or len(F_u) == 0:
            continue

        dists = []

        for r in recs:
            if r not in dm_map:
                continue
            for f in F_u:
                if f not in dm_map:
                    continue
                d = calculate_dm_distance(r, f)
                dists.append(d)

        if len(dists) == 0:
            continue

        total_novelty += np.mean(dists)
        total_users += 1

    return total_novelty / total_users if total_users > 0 else 0.0

In [52]:
indiv_nov_random = evaluate_individual_novelty_at_k(
    recommend_random,
    k=10
)

indiv_nov_popularity = evaluate_individual_novelty_at_k(
    recommend_popularity,
    k=10
)

indiv_nov_common_neighbors = evaluate_individual_novelty_at_k(
    recommend_common_neighbors,
    k=10
)

print("Random Baseline Individual Novelty@10:", indiv_nov_random)
print("Popularity Baseline Individual Novelty@10:", indiv_nov_popularity)
print("Common Neighbors Baseline Individual Novelty@10:", indiv_nov_common_neighbors)

Random Baseline Individual Novelty@10: 1.289524445246006
Popularity Baseline Individual Novelty@10: 1.1737697915784244
Common Neighbors Baseline Individual Novelty@10: 1.1350439037337448


##### 7.5.4 Get communities with louvain to evaluate community based metrics

In [55]:
# 1️⃣ Build UNDIRECTED weighted graph
G = nx.Graph()

# Count interaction frequency (edge weights)
edge_weights = (
    pos_train.groupby(["u", "v"])
    .size()
    .reset_index(name="weight")
)

for u, v, w in edge_weights.itertuples(index=False):
    if G.has_edge(u, v):
        G[u][v]["weight"] += w
    else:
        G.add_edge(u, v, weight=w)

# Detect communities on training graph
partition = community_louvain.best_partition(G)

# partition: dict {user -> community_id}
community_members = defaultdict(set)

for user, comm in partition.items():
    community_members[comm].add(user)


##### 7.5.5 Evaluate Community Diversity @ k 

In [44]:
def community_diversity(recommend_fn, k=10):
    community_scores = []

    for comm, members in community_members.items():
        R_c = set()

        for u in members:
            if u in user_vectors:
                R_c.update(recommend_fn(u, k=k))

        R_c = list(R_c)

        if len(R_c) < 2:
            continue

        dists = [
            calculate_dm_distance(i, j)
            for i in R_c for j in R_c
            if i != j
        ]

        if dists:
            community_scores.append(np.mean(dists))

    return np.mean(community_scores) if community_scores else 0.0

In [53]:
comm_div_random = community_diversity(
    recommend_random,
    k=10
)

comm_div_popularity = community_diversity(
    recommend_popularity,
    k=10
)

comm_div_common_neighbors = community_diversity(
    recommend_common_neighbors,
    k=10
)

print("Random Baseline Community Diversity@10:", comm_div_random)
print("Popularity Baseline Community Diversity@10:", comm_div_popularity)
print("Common Neighbors Community Diversity@10:", comm_div_common_neighbors)

Random Baseline Community Diversity@10: 1.2917204
Popularity Baseline Community Diversity@10: 1.1052775
Common Neighbors Community Diversity@10: 1.1477621


##### 7.5.6 Evaluate Community Novelty @ k

In [46]:
def community_novelty(recommend_fn, k=10):
    community_scores = []

    for comm, members in community_members.items():

        R_c = set()
        F_c = set()

        for u in members:
            if u in user_vectors:
                R_c.update(recommend_fn(u, k=k))
                F_c.update(train_edges.get(u, set()))

        R_c = list(R_c)
        F_c = list(F_c)

        if len(R_c) == 0 or len(F_c) == 0:
            continue

        dists = [
            calculate_dm_distance(i, j)
            for i in R_c for j in F_c
        ]

        if dists:
            community_scores.append(np.mean(dists))

    return np.mean(community_scores) if community_scores else 0.0

In [54]:
comm_nov_random = community_novelty(
    recommend_random,
    k=10
)

comm_nov_popularity = community_novelty(
    recommend_popularity,
    k=10
)

comm_nov_common_neighbors = community_novelty(
    recommend_common_neighbors,
    k=10
)

print("Random Baseline Community Novelty@10:", comm_nov_random)
print("Popularity Baseline Community Novelty@10:", comm_nov_popularity)
print("Common Neighbors Community Novelty@10:", comm_nov_common_neighbors)

Random Baseline Community Novelty@10: 1.2861842
Popularity Baseline Community Novelty@10: 1.311102
Common Neighbors Community Novelty@10: 1.0983262
